In [1]:
pip install pygame


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pygame, random, math, json, os
pygame.init()

WIDTH, HEIGHT = 1000, 800
screen = pygame.display.set_mode((WIDTH, HEIGHT))
clock = pygame.time.Clock()

STATS_FILE = "stats.json"
DDA_FILE = "dda_stats.json"

# ===== CONFIG =====
CONFIG = {
   "player_hp": 5,
   "player_speed": 4,

   "precision_damage": 3,
   "precision_range": 500,
   "precision_cd": 300,

   "shotgun_damage": 2,
   "shotgun_pellets": 6,
   "shotgun_range": 220,
   "shotgun_spread": 0.4,
   "shotgun_cd": 1200,

   "spawn_rate": 90,
   "enemy_hp_min": 4,
   "enemy_hp_max": 15,
   "enemy_hp_mult": 1,

   "chaser_speed": 2,
   "shooter_speed": 1.5,
   "boss_speed": 1.2,

   "shooter_hp_min": 2,
   "shooter_hp_max": 4,
   "shooter_distance": 250,

   "boss_hp": 1,
   "boss_duration": 30000,
   "boss_heal": 2,

   # BayesianDDA
   "dda_enabled": True,
   "dda_interval": 20000
}

font = pygame.font.SysFont(None, 28)
big_font = pygame.font.SysFont(None, 70)

# ===== STATS =====
def load_stats():
   if os.path.exists(STATS_FILE):
       with open(STATS_FILE, "r") as f:
           data = json.load(f)
       data.setdefault("shots", 0)
       data.setdefault("hits", 0)
       data.setdefault("next_shot_id", 0)
       data.setdefault("kills", 0)
       return data
   return {"shots": 0, "hits": 0, "next_shot_id": 0, "kills": 0}

def save_stats():
   with open(STATS_FILE, "w") as f:
       json.dump(stats, f)

def get_accuracy():
   if stats["shots"] == 0:
       return 0
   return stats["hits"] / stats["shots"] * 100

stats = load_stats()

# ===== BAYESIAN DDA =====
class BayesianDDA:
   def __init__(self, skill_range=(-3, 3), num_points=200, posterior=None):
       self.theta = [
           skill_range[0] + i * (skill_range[1] - skill_range[0]) / (num_points - 1)
           for i in range(num_points)
       ]

       if posterior:
           self.posterior = posterior
       else:
           self.posterior = [self.gaussian(x, 0, 1) for x in self.theta]
           self.normalize()

   def gaussian(self, x, mu, sigma):
       return math.exp(-0.5 * ((x - mu) / sigma) ** 2)

   def normalize(self):
       s = sum(self.posterior)
       if s == 0:
           self.posterior = [1 / len(self.posterior)] * len(self.posterior)
       else:
           self.posterior = [p / s for p in self.posterior]

   def likelihood(self, theta, difficulty, outcome):
       prob_win = 1 / (1 + math.exp(-(theta - difficulty)))
       return prob_win if outcome == 1 else 1 - prob_win

   def update(self, difficulty, outcome):
       self.posterior = [
           p * self.likelihood(t, difficulty, outcome)
           for p, t in zip(self.posterior, self.theta)
       ]
       self.normalize()

   def expected_skill(self):
       return sum(t * p for t, p in zip(self.theta, self.posterior))

   def choose_difficulty(self):
       return self.expected_skill()

def load_dda():
   if os.path.exists(DDA_FILE):
       with open(DDA_FILE, "r") as f:
           data = json.load(f)
       return BayesianDDA(posterior=data["posterior"])
   return BayesianDDA()

def save_dda():
   with open(DDA_FILE, "w") as f:
       json.dump({"posterior": dda.posterior}, f)

dda = load_dda()

def get_dynamic_params():
   skill = dda.choose_difficulty()
   pressure = max(0, min(1, (skill + 3) / 6))

   spawn_rate = int(CONFIG["spawn_rate"] * (1.25 - pressure * 0.55))
   enemy_hp_mult = CONFIG["enemy_hp_mult"] * (0.85 + pressure * 0.55)
   enemy_speed_mult = 0.85 + pressure * 0.45

   return {
       "spawn_rate": max(30, spawn_rate),
       "enemy_hp_mult": enemy_hp_mult,
       "enemy_speed_mult": enemy_speed_mult,
       "difficulty": skill
   }

def evaluate_dda_round():
   shots = stats["shots"] - state["round_shots"]
   hits = stats["hits"] - state["round_hits"]
   kills = stats["kills"] - state["round_kills"]
   hp_lost = state["round_hp"] - state["hp"]

   accuracy = hits / shots if shots > 0 else 0

   score = 0
   if accuracy >= 0.35: score += 1
   if kills >= 4: score += 1
   if hp_lost <= 1: score += 1
   if hp_lost >= 3: score -= 1

   outcome = 1 if score >= 2 else 0
   difficulty = dda.choose_difficulty()

   dda.update(difficulty, outcome)
   save_dda()

   state["round_shots"] = stats["shots"]
   state["round_hits"] = stats["hits"]
   state["round_kills"] = stats["kills"]
   state["round_hp"] = state["hp"]
   state["last_dda_update"] = pygame.time.get_ticks()

def size_from_hp(hp):
   return 10 + int(hp * 3.5)

def reset():
   return {
       "player": pygame.Rect(WIDTH//2, HEIGHT//2, 30, 30),
       "hp": CONFIG["player_hp"],
       "enemies": [],
       "bullets": [],
       "enemy_bullets": [],
       "weapon": "precision",
       "last_shot": 0,
       "frame": 0,
       "game_over": False,
       "start": pygame.time.get_ticks(),
       "boss_active": False,
       "boss_timer": 0,
       "last_boss_min": 0,
       "hit_shots": set(),

       "round_shots": stats["shots"],
       "round_hits": stats["hits"],
       "round_kills": stats["kills"],
       "round_hp": CONFIG["player_hp"],
       "last_dda_update": pygame.time.get_ticks(),
       "death_recorded": False
   }

state = reset()

cooldowns = {
   "precision": CONFIG["precision_cd"],
   "shotgun": CONFIG["shotgun_cd"]
}

def spawn_enemy():
   dyn = get_dynamic_params()

   side = random.choice(["top","bottom","left","right"])
   if side=="top": x,y=random.randint(0,WIDTH),-20
   elif side=="bottom": x,y=random.randint(0,WIDTH),HEIGHT
   elif side=="left": x,y=-20,random.randint(0,HEIGHT)
   else: x,y=WIDTH,random.randint(0,HEIGHT)

   if random.random()<0.3:
       hp = int(random.randint(CONFIG["shooter_hp_min"], CONFIG["shooter_hp_max"]) * dyn["enemy_hp_mult"])
       t="shooter"
   else:
       hp = int(random.randint(CONFIG["enemy_hp_min"], CONFIG["enemy_hp_max"]) * dyn["enemy_hp_mult"])
       t="chaser"

   size = size_from_hp(hp)

   state["enemies"].append({
       "rect": pygame.Rect(x,y,size,size),
       "hp": max(1, hp),
       "type": t,
       "cd": 0,
       "phase": 1
   })

def spawn_boss():
   state["enemies"].append({
       "rect": pygame.Rect(WIDTH//2-60,-120,120,120),
       "hp": CONFIG["boss_hp"],
       "type": "boss",
       "cd": 0,
       "phase": 1
   })
   state["boss_active"] = True
   state["boss_timer"] = pygame.time.get_ticks()

def can_shoot():
   now = pygame.time.get_ticks()
   if now - state["last_shot"] >= cooldowns[state["weapon"]]:
       state["last_shot"] = now
       return True
   return False

def shoot(target):
   if not can_shoot():
       return

   stats["shots"] += 1
   stats["next_shot_id"] += 1
   shot_id = stats["next_shot_id"]
   save_stats()

   px,py = state["player"].center
   ang = math.atan2(target[1]-py, target[0]-px)

   if state["weapon"] == "precision":
       state["bullets"].append({
           "x": px,"y": py,
           "dx": math.cos(ang)*12,
           "dy": math.sin(ang)*12,
           "dmg": CONFIG["precision_damage"],
           "range": CONFIG["precision_range"],
           "dist": 0,
           "shot_id": shot_id
       })
   else:
       for _ in range(CONFIG["shotgun_pellets"]):
           a = ang + random.uniform(-CONFIG["shotgun_spread"], CONFIG["shotgun_spread"])
           state["bullets"].append({
               "x": px,"y": py,
               "dx": math.cos(a)*8,
               "dy": math.sin(a)*8,
               "dmg": CONFIG["shotgun_damage"],
               "range": CONFIG["shotgun_range"],
               "dist": 0,
               "shot_id": shot_id
           })

def enemy_shoot(pos,target,speed=6):
   ang = math.atan2(target[1]-pos[1], target[0]-pos[0])
   state["enemy_bullets"].append({
       "x": pos[0],"y": pos[1],
       "dx": math.cos(ang)*speed,
       "dy": math.sin(ang)*speed
   })

def boss_attack(enemy):
   px,py = state["player"].center
   ex,ey = enemy["rect"].center

   if enemy["phase"] == 1:
       if enemy["cd"] <= 0:
           enemy_shoot((ex,ey),(px,py),7)
           enemy["cd"] = 50
   else:
       if enemy["cd"] <= 0:
           base = math.atan2(py-ey, px-ex)

           for i in range(-2,3):
               a = base + i*0.2
               state["enemy_bullets"].append({
                   "x":ex,"y":ey,
                   "dx":math.cos(a)*7,
                   "dy":math.sin(a)*7
               })

           for i in range(12):
               a = i*(math.pi*2/12)
               state["enemy_bullets"].append({
                   "x":ex,"y":ey,
                   "dx":math.cos(a)*5,
                   "dy":math.sin(a)*5
               })

           enemy["cd"] = 90

running = True
while running:
   clock.tick(60)
   state["frame"] += 1

   now = pygame.time.get_ticks()
   sec = (now - state["start"]) // 1000
   dyn = get_dynamic_params()

   for e in pygame.event.get():
       if e.type == pygame.QUIT:
           running = False

       if not state["game_over"]:
           if e.type == pygame.MOUSEBUTTONDOWN:
               shoot(pygame.mouse.get_pos())

           if e.type == pygame.KEYDOWN:
               if e.key == pygame.K_1: state["weapon"] = "precision"
               if e.key == pygame.K_2: state["weapon"] = "shotgun"
       else:
           if e.type == pygame.KEYDOWN:
               if e.key == pygame.K_r:
                   state = reset()
               if e.key == pygame.K_ESCAPE:
                   running = False

   if not state["game_over"]:

       if CONFIG["dda_enabled"] and now - state["last_dda_update"] >= CONFIG["dda_interval"]:
           evaluate_dda_round()

       if sec // 60 > state["last_boss_min"]:
           spawn_boss()
           state["last_boss_min"] = sec // 60

       if state["boss_active"] and now - state["boss_timer"] > CONFIG["boss_duration"]:
           state["boss_active"] = False

       keys = pygame.key.get_pressed()
       if keys[pygame.K_w]: state["player"].y -= CONFIG["player_speed"]
       if keys[pygame.K_s]: state["player"].y += CONFIG["player_speed"]
       if keys[pygame.K_a]: state["player"].x -= CONFIG["player_speed"]
       if keys[pygame.K_d]: state["player"].x += CONFIG["player_speed"]
       state["player"].clamp_ip(screen.get_rect())

       if not state["boss_active"]:
           if state["frame"] % dyn["spawn_rate"] == 0:
               spawn_enemy()

       new_en = []
       for en in state["enemies"]:
           ex,ey = en["rect"].center
           px,py = state["player"].center
           dx,dy = px-ex, py-ey
           dist = math.hypot(dx,dy)
           if dist: dx/=dist; dy/=dist

           if en["type"] == "boss":
               en["rect"].x += dx * CONFIG["boss_speed"] * dyn["enemy_speed_mult"]
               en["rect"].y += dy * CONFIG["boss_speed"] * dyn["enemy_speed_mult"]

               if en["hp"] <= CONFIG["boss_hp"]//2:
                   en["phase"] = 2

               boss_attack(en)
               en["cd"] -= 1

           elif en["type"] == "chaser":
               en["rect"].x += dx * CONFIG["chaser_speed"] * dyn["enemy_speed_mult"]
               en["rect"].y += dy * CONFIG["chaser_speed"] * dyn["enemy_speed_mult"]

               if en["rect"].colliderect(state["player"]):
                   state["hp"] -= 2
                   continue

           else:
               if dist > CONFIG["shooter_distance"]:
                   en["rect"].x += dx * CONFIG["shooter_speed"] * dyn["enemy_speed_mult"]
                   en["rect"].y += dy * CONFIG["shooter_speed"] * dyn["enemy_speed_mult"]
               else:
                   if en["cd"] <= 0:
                       enemy_shoot((ex,ey),(px,py))
                       en["cd"] = 90
                   else:
                       en["cd"] -= 1

           new_en.append(en)

       state["enemies"] = new_en

       new_b = []
       for b in state["bullets"]:
           b["x"] += b["dx"]
           b["y"] += b["dy"]
           b["dist"] += math.hypot(b["dx"], b["dy"])

           alive = True
           for en in state["enemies"]:
               if en["rect"].colliderect(pygame.Rect(b["x"],b["y"],4,4)):
                   en["hp"] -= b["dmg"]

                   if b["shot_id"] not in state["hit_shots"]:
                       stats["hits"] += 1
                       state["hit_shots"].add(b["shot_id"])
                       save_stats()

                   alive = False

                   if en["type"]=="boss" and en["hp"]<=0:
                       state["hp"] = min(CONFIG["player_hp"], state["hp"] + CONFIG["boss_heal"])
                       state["boss_active"] = False
                   break

           if alive and b["dist"] < b["range"]:
               new_b.append(b)

       state["bullets"] = new_b

       before = len(state["enemies"])
       state["enemies"] = [e for e in state["enemies"] if e["hp"] > 0]
       killed = before - len(state["enemies"])
       if killed > 0:
           stats["kills"] += killed
           save_stats()

       new_eb = []
       for b in state["enemy_bullets"]:
           b["x"] += b["dx"]
           b["y"] += b["dy"]

           if state["player"].colliderect(pygame.Rect(b["x"],b["y"],4,4)):
               state["hp"] -= 1
           else:
               new_eb.append(b)

       state["enemy_bullets"] = new_eb

       if state["hp"] <= 0:
           state["game_over"] = True
           if CONFIG["dda_enabled"] and not state["death_recorded"]:
               dda.update(dda.choose_difficulty(), 0)
               save_dda()
               state["death_recorded"] = True

   screen.fill((20,20,20))
   pygame.draw.rect(screen,(0,200,0),state["player"])

   for en in state["enemies"]:
       color = (255,120,0) if en["type"]=="boss" else (0,150,255) if en["type"]=="shooter" else (200,50,50)
       pygame.draw.rect(screen,color,en["rect"])

   for b in state["bullets"]:
       pygame.draw.circle(screen,(255,255,0),(int(b["x"]),int(b["y"])),2)

   for b in state["enemy_bullets"]:
       pygame.draw.circle(screen,(255,100,100),(int(b["x"]),int(b["y"])),3)

   pygame.draw.circle(screen,(255,255,255),pygame.mouse.get_pos(),8,1)

   screen.blit(font.render(f"HP: {state['hp']}",True,(255,255,255)),(10,10))
   screen.blit(font.render(f"Time: {sec}s",True,(255,255,255)),(10,30))
   screen.blit(font.render(f"Accuracy: {get_accuracy():.1f}%",True,(255,255,255)),(10,50))
   screen.blit(font.render(f"Shots: {stats['shots']}  Hits: {stats['hits']}  Kills: {stats['kills']}",True,(255,255,255)),(10,70))
   screen.blit(font.render(f"Weapon: {state['weapon']}",True,(255,255,255)),(10,90))

   screen.blit(font.render(f"DDA skill: {dda.expected_skill():.2f}",True,(255,255,255)),(10,120))
   screen.blit(font.render(f"Spawn time: {dyn['spawn_rate']}",True,(255,255,255)),(10,140))

   if state["game_over"]:
       screen.blit(big_font.render("GAME OVER",True,(255,50,50)),(250,350))
       screen.blit(font.render("R to restart",True,(255,255,255)),(330,420))
       screen.blit(font.render("ESC to quit",True,(255,255,255)),(330,460))

   pygame.display.flip()

pygame.quit()

pygame-ce 2.5.3 (SDL 2.30.12, Python 3.13.2)


2026-05-22 14:52:23.395 Python[26094:3224318] ApplePersistenceIgnoreState: Existing state will not be touched. New state will be written to /var/folders/k7/jd1_jz3s30g7q3dxpfq1_shr0000gn/T/org.python.python.savedState


: 